### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sf_permit_time",
    dataset_year="2025",
    domain_str="business & marketing",
    # Data Source
    dataset_source="GOV Website",
    original_dataset_source_download_link="https://data.sfgov.org/Housing-and-Buildings/Building-Permits/i98e-djp9",
    download_description="""
We re-collected the data similar to how the data from Kaggle was created.

We go to https://data.sfgov.org/Housing-and-Buildings/Building-Permits/i98e-djp9, then we set the query such that we only take data from 1st of January 2015 until 31st of December of 2025, and then download the data as a CSV file. Due to API limits, this must be done manually. Here is the link to the query: https://data.sfgov.org/Housing-and-Buildings/Building-Permits/i98e-djp9/explore/query/SELECT%0A%20%20%60permit_number%60%2C%0A%20%20%60permit_type%60%2C%0A%20%20%60permit_type_definition%60%2C%0A%20%20%60permit_creation_date%60%2C%0A%20%20%60block%60%2C%0A%20%20%60lot%60%2C%0A%20%20%60street_number%60%2C%0A%20%20%60street_number_suffix%60%2C%0A%20%20%60street_name%60%2C%0A%20%20%60street_suffix%60%2C%0A%20%20%60unit%60%2C%0A%20%20%60unit_suffix%60%2C%0A%20%20%60description%60%2C%0A%20%20%60status%60%2C%0A%20%20%60status_date%60%2C%0A%20%20%60filed_date%60%2C%0A%20%20%60issued_date%60%2C%0A%20%20%60completed_date%60%2C%0A%20%20%60first_construction_document_date%60%2C%0A%20%20%60approved_date%60%2C%0A%20%20%60structural_notification%60%2C%0A%20%20%60number_of_existing_stories%60%2C%0A%20%20%60number_of_proposed_stories%60%2C%0A%20%20%60voluntary_soft_story_retrofit%60%2C%0A%20%20%60fire_only_permit%60%2C%0A%20%20%60estimated_cost%60%2C%0A%20%20%60revised_cost%60%2C%0A%20%20%60existing_use%60%2C%0A%20%20%60existing_units%60%2C%0A%20%20%60proposed_use%60%2C%0A%20%20%60proposed_units%60%2C%0A%20%20%60plansets%60%2C%0A%20%20%60tidf_compliance%60%2C%0A%20%20%60existing_occupancy%60%2C%0A%20%20%60proposed_occupancy%60%2C%0A%20%20%60existing_construction_type%60%2C%0A%20%20%60existing_construction_type_description%60%2C%0A%20%20%60proposed_construction_type%60%2C%0A%20%20%60proposed_construction_type_description%60%2C%0A%20%20%60site_permit%60%2C%0A%20%20%60last_permit_activity_date%60%2C%0A%20%20%60application_submission_method%60%2C%0A%20%20%60adu%60%2C%0A%20%20%60primary_address_flag%60%2C%0A%20%20%60supervisor_district%60%2C%0A%20%20%60neighborhoods_analysis_boundaries%60%2C%0A%20%20%60zipcode%60%2C%0A%20%20%60location%60%2C%0A%20%20%60point_source%60%2C%0A%20%20%60reroof%60%2C%0A%20%20%60record_id%60%2C%0A%20%20%60data_as_of%60%2C%0A%20%20%60data_loaded_at%60%0AWHERE%0A%20%20%60approved_date%60%0A%20%20%20%20BETWEEN%20%222015-01-01T16%3A13%3A34%22%20%3A%3A%20floating_timestamp%0A%20%20%20%20AND%20%222025-12-31T16%3A13%3A34%22%20%3A%3A%20floating_timestamp%0AORDER%20BY%20%60approved_date%60%20ASC%20NULL%20LAST/page/filter

We save the file in the root dir as Building_Permits_20260205.csv
mkdir -p local-data-warehouse/sf_permit_time && mv Building_Permits_20260205.csv local-data-warehouse/sf_permit_time
""",
    # References
    academic_reference_bibtex=r"""@misc{SanFrancisco2026BuildingPermits,
  author = {{City and County of San Francisco}},
  title  = {Building Permits},
  year   = {2026},
  howpublished = {\url{https://data.sfgov.org/Housing-and-Buildings/Building-Permits/i98e-djp9/about_data}},
  note   = {DataSF Open Data Portal dataset, Accessed: 2026-02-05}
}
""",
    academic_reference_bibtex_key="SanFrancisco2026BuildingPermits",
    license="Open Data Commons Public Domain Dedication and License",
    data_tags=["Non-IID","Temporal"],
    curation_comments="""
We simulate the task of predicting the days (in float) it takes to issue the permit. We add the special use case, that we assume the model is only used to predict for permits that take longer than one day to be issued. We do this, as waiting for one day seems very reasonable. Plus, the data contains several unresolvable data errors when the permit was issued on the same day it was filed.

- ALthough we donwloaded data
- We drop miscellaneous permits (those that start with "M") as they are not of interest for our task. Compared to normal permits, these are usually automatically or very quickly approved and thus are not relevant to predict the time it takes to issue the permit. Moreover, the represent a significant distribution shift compared to the rest of the data.
- We drop the permit type ordinal encoding and the creation date of the permit in the tracking system as other dates are more accurate.
- A large number of descriptions are from standard phrases (they appear in the same way multiple times). so we add a new column that indicates whether the description is from a standard phrase or not through a categorical variable. We define a standard phrase as a description that appears more than 100 times in the dataset.
- The 'Current Status' is the last status update of the permit. Note, that for miscellaneous permits, the permit is never completed. We filter to permits that have been issued for our task. Thus, we select all permits that are issued or completed.
- There are several features that have almost no information (up to only 20 non nan values). We keep this and let the pipeline decide what to do with them.
- We only keep on permit per primary address (filter based on Primary Address Flag) following https://data.sfgov.org/Housing-and-Buildings/Building-Permits-Deduplicated-on-Primary-Address/f2jc-ivnc
- We drop all permits without a location, as every permit requires a location by definition, thus these are likely data errors.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DaysToIssue",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="Filed Date"
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "Building_Permits_20260205.csv")
# Things to check
# - address unique cases as on the website (This is a filtered view of the Building Permits dataset. This dataset only shows building permits where the 'Primary Address Flag' is equal to "Y", indicating it is the primary address)

df["DaysToIssue"] = (pd.to_datetime(df["Issued Date"]) - pd.to_datetime(df["Filed Date"])).dt.total_seconds() / pd.Timedelta(days=1).total_seconds()

df = df[df["DaysToIssue"] >= 1]
df = df[~df["Permit Number"].str.startswith("M")]
df = df[df["Current Status"].isin(["issued", "complete"])]
df = df[df["Primary Address Flag"] == "Y"]
df = df[~df["Location"].isna()]

df["DescriptionIsStandardPhrase"] = df["Description"].isin(df["Description"].value_counts(dropna=False)[df["Description"].value_counts(dropna=False) > 100].index)
df["Location_Latitude"] = df["Location"].apply(lambda x: str(x).split("(")[-1].split(" ")[0]).astype(float)
df["Location_Longitude"] = df["Location"].apply(lambda x: str(x).split("(")[-1].split(" ")[-1][:-1]).astype(float)

df = df.drop(columns=[
    "Permit Creation Date", # Just a log date from the system.
    "Permit Type", # Ordinal encoding of "Permit Type Definition"
    "Current Status", # repeat information from other columns, also leaks into the future
    "Current Status Date", # leaks into the future and the target
    "Completed Date", # completed happens after issuing, so this is after our task and outside of the control of the government body that issues the permit. Moreover, it leaks the target variable.
    "First Construction Document Date", # same as above
    "approved_date", # often the same value as issued date depending on the type of permit, thus leakage
    "Issued Date", # Target leakage
    "Revised Cost", # Also only available after the permit is issued most likely (after project review)
    "Existing Construction Type", # Ordinal encoding of "Existing Construction Type Description"
    "Proposed Construction Type", # Ordinal encoding of "Proposed Construction Type Description"
    "Last Permit Activity Date", # target leakage
    "ADU", # Deprecated column from 2025 onwards
    "Primary Address Flag", # constant after filter above
    "Record ID", # Unique ID
    "data_as_of", # metadata from logging system
    "data_loaded_at", # metadata from logging system
    "Location", # resolved above
    "Permit Number", # meaningless ID given other info
])

as_cat_type = [
    "Permit Type Definition",
    "DescriptionIsStandardPhrase",
    "Structural Notification",
    "Voluntary Soft-Story Retrofit",
    "Fire Only Permit",
    "TIDF Compliance",
    "Existing Construction Type Description",
    "Proposed Construction Type Description",
    "Site Permit",
    "Application Submission Method",
    "supervisor_district",
    "neighborhoods_analysis_boundaries",
    "point_source",
    "reroof",
]
as_string_type = [
    "Block",
    "Lot",
    "Street Number",
    "Street Number Suffix",
    "Street Name",
    "Street Suffix",
    "Unit",
    "Unit Suffix",
    "Description",
    "Existing Use",
    "Proposed Use",
    "Existing Occupancy",
    "Proposed Occupancy",
    "Zipcode",
]
as_datetime_type = [
    "Filed Date"
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

for c in as_datetime_type:
    df[c] = pd.to_datetime(df[c])

df[as_cat_type] = df[as_cat_type].astype("category")

# Log scale the target
df["DaysToIssue"] = np.log(df["DaysToIssue"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

/tmp/ipykernel_614421/2993407991.py:4: DtypeWarning: Columns (19,23,32,35,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_mold.path / "Building_Permits_20260205.csv")


/tmp/ipykernel_614421/2993407991.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DaysToIssue"] = (pd.to_datetime(df["Issued Date"]) - pd.to_datetime(df["Filed Date"])).dt.total_seconds() / pd.Timedelta(days=1).total_seconds()


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 116,954
Columns: 38
Use sampling: False (sample size: 116,954)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Filed Date', 'Description', 'Location_Latitude', 'Location_Longitude', 'Estimated Cost', 'Block', 'Street Number', 'Proposed Occupancy', 'Existing Occupancy', 'Street Name']
Rows remaining as candidates after top-10 filter: 36 (of 116,954)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 18 (0.02% of dataset)
Get column duplicates...


Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Permit Type Definition,Block,Lot,Street Number,Street Number Suffix,Street Name,Street Suffix,Unit,Unit Suffix,Description,Filed Date,Structural Notification,Number of Existing Stories,Number of Proposed Stories,Voluntary Soft-Story Retrofit,Fire Only Permit,Estimated Cost,Existing Use,Existing Units,Proposed Use,Proposed Units,Plansets,TIDF Compliance,Existing Occupancy,Proposed Occupancy,Existing Construction Type Description,Proposed Construction Type Description,Site Permit,Application Submission Method,supervisor_district,neighborhoods_analysis_boundaries,Zipcode,point_source,reroof,DaysToIssue,DescriptionIsStandardPhrase,Location_Latitude,Location_Longitude
0,otc alterations permit,3602,036,225.0,<NA>,Hartford,St,0.0,<NA>,"to comply with nov#201390061, construction of new egress stair from upper apartment that had been illegally removed by previous owner. removal of non permitted space, restoration of original exterior wall at location of non permitted additions, new rear deck.",2016-09-23 14:26:18,Y,4.0,4.0,NaN,NaN,19500.0,apartments,4.0,apartments,4.0,2.0,NaN,R-2,R-2,wood frame (5),wood frame (5),NaN,in-house,8.0,Castro/Upper Market,94114.0,eas_address_point,NaN,1.095633,False,-122.433383,37.758930
1,otc alterations permit,2019,016,2350.0,<NA>,Noriega,St,<NA>,<NA>,"install one utility transformer, 1 switchboard assembly, 4 150 kw btc power units, 4 150 kw btc dispensers.",2019-04-09 12:53:35,NaN,0.0,0.0,NaN,NaN,100000.0,parking lot,0.0,parking lot,0.0,2.0,NaN,S-2,S-2,constr type 1,constr type 1,NaN,in-house,4.0,Sunset/Parkside,94122.0,eas_address_point,NaN,4.354326,False,-122.488820,37.754128
2,otc alterations permit,1291,034,617.0,<NA>,Belvedere,St,<NA>,<NA>,"top unit: proposed interior remodel, consisting of expanding kitchen, & reconfigure existing bathroom.",2020-02-03 11:04:15,NaN,3.0,3.0,NaN,NaN,150000.0,2 family dwelling,2.0,2 family dwelling,2.0,2.0,NaN,R-3,R-3,wood frame (5),wood frame (5),NaN,in-house,8.0,Inner Sunset,94117.0,eas_address_point,NaN,2.075805,False,-122.448001,37.761444
3,otc alterations permit,1931,035C,621.0,<NA>,Lawton,St,<NA>,<NA>,voluntary foundation - bolt house & sheath cripple walls as per details,2015-04-03 15:54:59,NaN,2.0,2.0,NaN,NaN,8000.0,1 family dwelling,1.0,1 family dwelling,1.0,2.0,NaN,R-3,R-3,wood frame (5),wood frame (5),NaN,in-house,7.0,Inner Sunset,94122.0,eas_address_point,NaN,1.011310,False,-122.469563,37.758040
4,otc alterations permit,3720,009,415.0,<NA>,Mission,St,<NA>,<NA>,non rated stair connecting floors 34 and 35. ref . 2017-09-08-7306,2017-09-08 07:46:02,NaN,63.0,63.0,NaN,NaN,120000.0,office,0.0,office,0.0,2.0,NaN,B,B,constr type 1,constr type 1,NaN,in-house,6.0,Financial District/South Beach,94105.0,eas_address_point,NaN,1.170319,False,-122.397097,37.789934


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,TIDF Compliance,category,116953.0,100.00,1.0,P
1,Voluntary Soft-Story Retrofit,category,116943.0,99.99,1.0,Y
2,reroof,category,115306.0,98.59,1.0,Y
3,Site Permit,category,113361.0,96.93,1.0,Y
4,Structural Notification,category,109395.0,93.54,1.0,Y
5,Fire Only Permit,category,102457.0,87.60,1.0,Y
6,Proposed Construction Type Description,category,5865.0,5.01,5.0,"wood frame (5), constr type 1, constr type 3, constr type 2, constr type 4"
7,Existing Construction Type Description,category,4975.0,4.25,5.0,"wood frame (5), constr type 1, constr type 3, constr type 2, constr type 4"
8,Permit Type Definition,category,168.0,0.14,8.0,"otc alterations permit, additions alterations or repairs, sign - erect, demolitions, new construction wood frame, wall or painted sign, new construction, grade or quarry or fill or excavate"
9,Application Submission Method,category,0.0,0.00,3.0,"in-house, website, epr website"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Number of Existing Stories,112432.0,6.862779,1.024347e+01,0.000000,6.700000e+01
Number of Proposed Stories,111410.0,6.933902,1.025844e+01,0.000000,1.180000e+02
Estimated Cost,116713.0,246251.141330,3.856691e+06,0.000000,5.379586e+08
Existing Units,95226.0,18.586510,8.123818e+01,0.000000,1.907000e+03
Proposed Units,95775.0,19.772613,8.154362e+01,0.000000,1.907000e+03
Plansets,116877.0,1.779734,6.258673e-01,0.000000,9.000000e+00
DaysToIssue,116954.0,3.246426,1.716170e+00,0.000012,8.139683e+00
Location_Latitude,116954.0,-122.429255,2.982557e-02,-122.510735,-1.223629e+02
Location_Longitude,116954.0,37.769456,2.412744e-02,37.708201,3.782981e+01


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                                 rank                                                                        
Application Submission Method          1                                                                in-house   
                                       2                                                                 website   
                                       3                                                             epr website   
Block                                  1                                                                    3708   
                                       2                                                                    3721   
                                       3                                                                    3713   
                                       4                                                                    3710   
                                       5                                                                    3707   
Description                            1                                                               reroofing   
                                       2     soft story retrofit per sfebc chapter 4d engineering criteria 20...   
                                       3     upgrade existing fire alarm system to comply with sffc section 1...   
                                       4                                                              re-roofing   
                                       5                                                              reroofing.   
DescriptionIsStandardPhrase            1                                                                   False   
                                       2                                                                    True   
Existing Construction Type Description 1                                                          wood frame (5)   
                                       2                                                           constr type 1   
                                       3                                                           constr type 3   
                                       4                                                                    <NA>   
                                       5                                                           constr type 2   
Existing Occupancy                     1                                                                     R-3   
                                       2                                                                     R-2   
                                       3                                                                       B   
                                       4                                                                    <NA>   
                                       5                                                                     B,M   
Existing Use                           1                                                       1 family dwelling   
                                       2                                                              apartments   
                                       3                                                                  office   
                                       4                                                       2 family dwelling   
                                       5                                                            retail sales   
Filed Date                             1                                                     2020-06-12 19:52:07   
                                       2                                                     2015-11-03 11:55:38   
                                       3                                                     2015-06-09 08:59:11   
                                       4       

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.15,-3.086,2.945,1.104,log,509349.5,82741244.1,exponential


## Task Curation

In [9]:
# Filter to year 2025
df_2025 = df.copy()

# Create a year-month column for grouping
df_2025["year_month"] = df_2025[task_mold.time_on].dt.to_period("Y")

# 1) Total number of samples per month
monthly_totals = (
    df_2025
    .groupby("year_month")
    .size()
    .rename("total_samples")
)

# Optional: sort by month and convert PeriodIndex to timestamp (month start)
result = monthly_totals.sort_index()
result.index = result.index.to_timestamp()
result

year_month
2015-01-01    11958
2016-01-01    11840
2017-01-01    12006
2018-01-01    12786
2019-01-01    12194
2020-01-01    10636
2021-01-01     9915
2022-01-01     9449
2023-01-01     9063
2024-01-01     8532
2025-01-01     8575
Freq: YS-JAN, Name: total_samples, dtype: int64

In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)

splits = {}

test_years = [
    "2020",
    "2021",
    "2022",
    "2023",
    "2024",
    "2025",
]
for i, month in enumerate(test_years):
    ref_date  = pd.Timestamp(month)
    train_index = df[
        df[task_mold.time_on] < ref_date
    ].index
    test_index = df[
        (df[task_mold.time_on].dt.year == ref_date.year)
    ].index
    splits[i] = {
        0: (train_index.tolist(), test_index.tolist())
    }

for s in splits:
    train_index, test_index = splits[s][0]
    print(f"Split {s}: Train size: {len(train_index)}, Test size: {len(test_index)}")
    assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

The official data is updated daily but has not enough data per day to create large enough test splits. We opt for simulating a model that is refit every year to obtain a robust test set instead.
This introduces the unrealistic downside of data shift across a month that would not exist in a real-world model. We create 5 test splits by 2020-2025 as test year. For each test split, we use all data before the test month as training data.
""",
    splits=splits,
    time_horizon=1,
    time_horizon_unit="years",
)

Split 0: Train size: 60784, Test size: 10636
Split 1: Train size: 71420, Test size: 9915
Split 2: Train size: 81335, Test size: 9449
Split 3: Train size: 90784, Test size: 9063
Split 4: Train size: 99847, Test size: 8532
Split 5: Train size: 108379, Test size: 8575


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to sf_permit_time/019d738d-80c6-71a6-99f0-e3e0955ce2d0


019d738d-80c6-71a6-99f0-e3e0955ce2d0
1f766752fbab7d592f90edb006afbeb17a39eb0c5cb06d66777968c4d495750c
